In [1]:
#import libraries
from pathlib import Path
import numpy as np
import pandas as pd

In [2]:
#read the data, dtype = str to give Pandas no room for interpretation
sf_raw = pd.read_csv("Sozioökonomische Faktoren 21.03.01.csv", sep = ";", encoding = "utf-8", dtype = str)

sf_raw["VALUE"] = pd.to_numeric(sf_raw["VALUE"])

sf_raw.head()

,PERIOD_REF,PERIOD_COMP,CODE_REGION,REGION,INDICATORS,UNIT_MES,VALUE,STATUS
0,2019,NaN,CH,Schweiz,Ind_01_01,UM_6,8606033.0,A
1,2019,NaN,1,Aeugst am Albis,Ind_01_01,UM_6,1981.0,A
2,2019,NaN,2,Affoltern am Albis,Ind_01_01,UM_6,12303.0,A
3,2019,NaN,3,Bonstetten,Ind_01_01,UM_6,5572.0,A
4,2019,NaN,4,Hausen am Albis,Ind_01_01,UM_6,3751.0,A


We have 2173 different regions, 41 different values in 4 different years. About 5% of values are missing for different communities.

In [3]:
sf_raw["PERIOD_REF"].value_counts()

PERIOD_REF
2019         49979
2018         19557
2004/2009     8692
1979/1985     4346
2010/2019     2173
2016          2173
2020          2173
Name: count, dtype: int64

In [4]:
sf_raw["STATUS"].value_counts()

STATUS
A    84945
M     2842
Q     1306
Name: count, dtype: int64

M and Q can't be used see appendix

In [5]:
print(sf_raw["CODE_REGION"].nunique())
print(sf_raw["INDICATORS"].nunique())

2173
41


In [6]:
indicators = pd.read_excel("Sozioökonomische Faktoren 21.03.01-APPENDIX.ods", sheet_name= "INDICATORS", header= 1)
indicators = indicators[["INDICATORS", "EN"]]
indicators

,INDICATORS,EN
0,Ind_01_01,Population - Residents
1,Ind_01_02,Population - Change in %
2,Ind_01_03,Population - Population density
3,Ind_01_04,Population - Share of age group 0-19 years
4,Ind_01_05,Population - Share of age group 20-64 years
5,Ind_01_06,Population - Share of age group 65+ years
6,Ind_01_08,Population - Foreign nationals
7,Ind_01_09,Population - Crude marriage rate
8,Ind_01_10,Population - Crude divorce rate
9,Ind_01_11,Population - Crude birth rate


In [7]:
units = pd.read_excel("Sozioökonomische Faktoren 21.03.01-APPENDIX.ods", sheet_name= "UNIT_MES", header= 1)
units = units.rename(columns={"Unnamed: 0": "UNIT_MES"}) #renameing the first column
units = units[["UNIT_MES", "EN"]]
units

,UNIT_MES,EN
0,UNIT_MES,Unit
1,UM_2,%
2,UM_3,Inhabitants per km² of total area
3,UM_4,‰
4,UM_6,in persons
5,UM_9,Business establishments
6,UM_10,in ha
7,UM_11,in m²
8,UM_12,per 1000 residents
9,UM_16,Absolute number


In [8]:
status = pd.read_excel("Sozioökonomische Faktoren 21.03.01-APPENDIX.ods", sheet_name= "STATUS", header= 1)
status = status[["STATUS", "EN"]]
status

,STATUS,EN
0,A,Normal value
1,M,Not indicated because data was not important o...
2,Q,Not indicated to protect confidentiality


In [9]:
sf_raw.duplicated().sum()#check_for_duplicates


np.int64(0)

In [10]:
# check only for one Indicator only 1 year and one unit
meta_sf = sf_raw.groupby("INDICATORS").agg(
    year = ("PERIOD_REF", "first"),
    amount_year = ("PERIOD_REF", "nunique"),
    UNIT_MES = ("UNIT_MES",  "first"),
    amount_units = ("UNIT_MES", "nunique")
)

meta_sf

,year,amount_year,UNIT_MES,amount_units
INDICATORS,,,,
Ind_01_01,2019,1,UM_6,1
Ind_01_02,2010/2019,1,UM_2,1
Ind_01_03,2019,1,UM_3,1
Ind_01_04,2019,1,UM_2,1
Ind_01_05,2019,1,UM_2,1
Ind_01_06,2019,1,UM_2,1
Ind_01_08,2019,1,UM_2,1
Ind_01_09,2019,1,UM_4,1
Ind_01_10,2019,1,UM_4,1


In [11]:
meta_sf = meta_sf.merge(indicators, on = "INDICATORS", how = "left")
meta_sf = meta_sf.merge(units, on = "UNIT_MES", how = "left")
meta_sf = meta_sf[["INDICATORS", "year", "EN_x", "EN_y"]]
meta_sf.columns = ["Indicators", "year", "ind_description", "ind_unit"]
meta_sf


,Indicators,year,ind_description,ind_unit
0,Ind_01_01,2019,Population - Residents,in persons
1,Ind_01_02,2010/2019,Population - Change in %,%
2,Ind_01_03,2019,Population - Population density,Inhabitants per km² of total area
3,Ind_01_04,2019,Population - Share of age group 0-19 years,%
4,Ind_01_05,2019,Population - Share of age group 20-64 years,%
5,Ind_01_06,2019,Population - Share of age group 65+ years,%
6,Ind_01_08,2019,Population - Foreign nationals,%
7,Ind_01_09,2019,Population - Crude marriage rate,‰
8,Ind_01_10,2019,Population - Crude divorce rate,‰
9,Ind_01_11,2019,Population - Crude birth rate,‰


In [12]:
sf_raw.isna().sum() # checking for missing values

PERIOD_REF         0
PERIOD_COMP    84747
CODE_REGION        0
REGION             0
INDICATORS         0
UNIT_MES           0
VALUE           4148
STATUS             0
dtype: int64

In [13]:
missing = sf_raw[sf_raw["VALUE"].isna()]
missing["STATUS"].value_counts()


STATUS
M    2842
Q    1306
Name: count, dtype: int64

In [14]:
sf_raw[sf_raw["STATUS"] == "M"]["INDICATORS"].value_counts() # where M comes from

INDICATORS
Ind_14_08    1186
Ind_14_07     529
Ind_14_10     502
Ind_14_05     311
Ind_14_06     125
Ind_14_09      69
Ind_14_01      45
Ind_14_03      37
Ind_14_02      29
Ind_14_04       9
Name: count, dtype: int64

In [15]:
sf_raw[sf_raw["STATUS"] == "Q"]["INDICATORS"].value_counts() #where q comes from


INDICATORS
Ind_11_01    480
Ind_06_07    297
Ind_06_09    163
Ind_06_08    150
Ind_06_03     96
Ind_06_05     57
Ind_06_04     36
Ind_06_10     21
Ind_06_06      6
Name: count, dtype: int64

In [16]:
# taking out swiss Data
switzerland = sf_raw[sf_raw["CODE_REGION"] == "CH"].copy()
communities = sf_raw[sf_raw["CODE_REGION"] != "CH"].copy()

In [17]:
communities = communities[["CODE_REGION", "REGION", "PERIOD_REF", "INDICATORS", "VALUE"]]
communities.columns = ["Code_Community", "Community", "Year", "Indicators", "Value" ]
communities.head()

,Code_Community,Community,Year,Indicators,Value
1,1,Aeugst am Albis,2019,Ind_01_01,1981.0
2,2,Affoltern am Albis,2019,Ind_01_01,12303.0
3,3,Bonstetten,2019,Ind_01_01,5572.0
4,4,Hausen am Albis,2019,Ind_01_01,3751.0
5,5,Hedingen,2019,Ind_01_01,3778.0


In [18]:
new_names = dict(zip(indicators["INDICATORS"], indicators["EN"]))
wide = communities.pivot(index="Community", columns="Indicators", values="Value")
wide = wide.rename(columns=new_names)
wide

Indicators,Population - Residents,Population - Change in %,Population - Population density,Population - Share of age group 0-19 years,Population - Share of age group 20-64 years,Population - Share of age group 65+ years,Population - Foreign nationals,Population - Crude marriage rate,Population - Crude divorce rate,Population - Crude birth rate,...,Voter shares of selected parties (National Council elections) - FDP/PLR,Voter shares of selected parties (National Council elections) - CVP,Voter shares of selected parties (National Council elections) - SP,Voter shares of selected parties (National Council elections) - SVP,Voter shares of selected parties (National Council elections) - EVP/CSP,Voter shares of selected parties (National Council elections) - GLP,Voter shares of selected parties (National Council elections) - BDP,Voter shares of selected parties (National Council elections) - PdA/Sol.,Voter shares of selected parties (National Council elections) - GPS,Voter shares of selected parties (National Council elections) - Small right-wing parties
Community,,,,,,,,,,,,,,,,,,,,,
Aadorf,9047.0,11.912420,453.711133,20.404554,60.362551,19.232895,15.839505,5.750940,2.433090,11.723070,...,11.398277,15.480451,10.728960,37.415507,3.412856,7.912525,2.080848,NaN,9.589132,1.981445
Aarau,21773.0,11.673591,1764.424635,17.383916,64.414642,18.201442,20.938777,5.868897,2.310589,11.229465,...,16.680704,5.660684,26.428113,16.347677,3.623116,10.853120,2.539040,NaN,14.680225,0.629738
Aarberg,4632.0,13.307241,584.110971,19.624352,59.045769,21.329879,12.888601,4.751620,1.943844,9.935205,...,9.919586,0.993350,14.825120,33.668160,3.817580,9.844459,12.190100,0.303292,10.256268,1.811403
Aarburg,8376.0,19.657143,1899.319728,20.475167,65.222063,14.302770,44.138013,4.706450,3.499668,10.619683,...,11.323108,5.880994,20.964612,34.902522,3.095260,6.430749,2.642521,NaN,10.519265,1.723182
Aarwangen,4620.0,10.526316,466.666667,20.541126,60.129870,19.329004,15.909091,5.010347,1.742730,11.327742,...,11.622443,1.259359,13.289057,38.645406,2.562576,9.059867,8.677673,0.137840,9.006610,2.048808
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zwieselberg,329.0,21.851852,133.739837,27.051672,56.534954,16.413374,4.863222,3.025719,0.000000,6.051437,...,3.261243,0.926880,9.921044,40.370752,4.600069,4.874700,9.818057,0.171644,13.422588,6.385170
Zwingen,2453.0,12.677997,532.104122,20.423971,64.329393,15.246637,24.419079,6.595218,2.061006,11.541632,...,11.336769,20.915429,19.854501,33.404062,2.606851,2.606851,0.818430,NaN,8.396484,NaN
Zwischbergen,74.0,-7.500000,0.860165,2.702703,81.081081,16.216216,27.027027,0.000000,13.245033,0.000000,...,1.673640,37.238494,1.255230,58.577406,0.000000,0.000000,NaN,NaN,1.255230,NaN


In [19]:
codebook = sf_raw[["INDICATORS", "PERIOD_REF", "UNIT_MES"]].drop_duplicates()

codebook = codebook.merge(indicators[["INDICATORS", "EN"]], on="INDICATORS", how="left")
codebook = codebook.merge(units[["UNIT_MES", "EN"]], on="UNIT_MES", how="left", suffixes=("", "_unit"))

codebook

,INDICATORS,PERIOD_REF,UNIT_MES,EN,EN_unit
0,Ind_01_01,2019,UM_6,Population - Residents,in persons
1,Ind_01_02,2010/2019,UM_2,Population - Change in %,%
2,Ind_01_03,2019,UM_3,Population - Population density,Inhabitants per km² of total area
3,Ind_01_08,2019,UM_2,Population - Foreign nationals,%
4,Ind_01_04,2019,UM_2,Population - Share of age group 0-19 years,%
5,Ind_01_05,2019,UM_2,Population - Share of age group 20-64 years,%
6,Ind_01_06,2019,UM_2,Population - Share of age group 65+ years,%
7,Ind_01_09,2019,UM_4,Population - Crude marriage rate,‰
8,Ind_01_10,2019,UM_4,Population - Crude divorce rate,‰
9,Ind_01_11,2019,UM_4,Population - Crude birth rate,‰


In [20]:
print(wide.isna().sum())

Indicators
Population - Residents                                                                         0
Population - Change in %                                                                       0
Population - Population density                                                                0
Population - Share of age group 0-19 years                                                     0
Population - Share of age group 20-64 years                                                    0
Population - Share of age group 65+ years                                                      0
Population - Foreign nationals                                                                 0
Population - Crude marriage rate                                                               0
Population - Crude divorce rate                                                                0
Population - Crude birth rate                                                                  0
Population - Crude 

In [23]:
wide.to_csv("socioeconomics_factors.csv", index = True)
meta_sf.to_csv("socioeconomics_factors_APPENDIX.csv", index = True)